Status:
Currently, I have pulled the correct dataset (mrnet-acl) down from Hugging Face and into train/test splits. I have been able to send a numpy array to MedGemma and get a repsponse.

GenAI: I used Claude and HuggingFaceAI to learn how to download a set of npz files from HuggingFaceHub. I used Gemini to draft code to load the model and run inference on MedGamma. I used Gemini to identify possible solutions to an Out of Memory issue. I used Gemini to convert a numpy array to a Hugging Face dataset with train/test splits. I used Claude to learn how to convert a PyArrow table to a HF dataset. I used Gemini to learn that I should use a HF dataset as an intermediary between a PyArrow array and MedGemma.

Reference for fine tuning MedGamma: https://github.com/google-health/medgemma/blob/main/notebooks/fine_tune_with_hugging_face.ipynb

In [36]:
!pip install bitsandbytes
!pip install accelerate

In [37]:
from datasets import load_dataset
from huggingface_hub import login
from huggingface_hub import hf_hub_download
from datasets import Dataset, concatenate_datasets
from transformers import pipeline
import numpy as np
from huggingface_hub import snapshot_download
import glob
from torchvision import transforms
from torch import Tensor
from sklearn.preprocessing import normalize
from huggingface_hub import snapshot_download
import pyarrow.parquet as pq
import pandas as pd
import pyarrow as pa

In [38]:
from huggingface_hub import login
login()    # now prompt for the new one

Get files from Drive

Download from Hugging Face

In [39]:
def download_one_example(repo_id, filepath):
  filepath1 = hf_hub_download(
      repo_id=repo_id,
      filename=filepath,
      repo_type="dataset"
  )
  ds = pq.read_table(filepath1)
  outputs = ds["label"]
  ds = ds.drop("label")

  ds = Dataset(ds)

  return ds, outputs

In [40]:
# Example: use AUMLProject/acl-mri-dataset and data/cai2r_batch_0.npz
# OR AUMLProject/acl-mri-dataset and data/mrnet_batch_9.npz
# OR AUMLProject/fastmri-knee-singlecoil-rss and data/file1000001.npz
example, output = download_one_example("AUMLProject/mrnet-acl", "data/train-00000-of-00029.parquet")
print(example)

Dataset({
    features: ['sagittal', 'coronal', 'axial'],
    num_rows: 39
})


Download the mrnet_batch files from hugging face as a train and test set

In [41]:
# TODO - This currently returns all_ouputs as a list of ChunkedArrays -> need to convert to an actual single chunked array (or maybe we can get away with processing a list of chunked arrays?)
# Download with 80% train and 20% test from Hugging Face

from datasets import load_dataset, DatasetDict

def download_train_test_datasets(seed=42, n_files=None):
    """Use MRNet's official train and validation splits (no random splitting)."""
    if n_files is None:
        train_ds = load_dataset("AUMLProject/mrnet-acl", split="train")
    else:
        data_files = [f"data/train-{i:05d}-of-00029.parquet" for i in range(n_files)]
        train_ds = load_dataset(
            "AUMLProject/mrnet-acl",
            data_files=data_files,
            split="train",
            verification_mode="no_checks",
        )

    # Official held-out set
    test_ds = load_dataset("AUMLProject/mrnet-acl", split="validation")

    ds = DatasetDict({"train": train_ds, "test": test_ds})

    import pandas as pd
    print(f"Train size: {len(ds['train'])}")
    print(f"Test size:  {len(ds['test'])}")
    print(f"Train label dist: {pd.Series(ds['train']['label']).value_counts().to_dict()}")
    print(f"Test label dist:  {pd.Series(ds['test']['label']).value_counts().to_dict()}")
    return ds


In [42]:

ds = download_train_test_datasets(n_files=3)

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Train size: 117
Test size:  120
Train label dist: {0: 103, 1: 14}
Test label dist:  {0: 66, 1: 54}


In [43]:
print(ds) # Each feature of the train data is a different viewpoint of the knee (axial, etc)

DatasetDict({
    train: Dataset({
        features: ['sagittal', 'coronal', 'axial', 'label'],
        num_rows: 117
    })
    test: Dataset({
        features: ['sagittal', 'coronal', 'axial', 'label'],
        num_rows: 120
    })
})


In [44]:
# Get the underlying arrow table and pull column data directly
test_table = ds["test"].data.table

row_idx = 0
for view in ["sagittal", "coronal", "axial"]:
    # Get the raw nested list and convert to numpy
    raw = test_table.column(view)[row_idx].as_py()
    arr = np.array(raw)
    print(f"{view}: shape {arr.shape}, dtype {arr.dtype}")

label = test_table.column("label")[row_idx].as_py()
print(f"label: {label}")

sagittal: shape (27, 256, 256), dtype float16
coronal: shape (25, 256, 256), dtype float16
axial: shape (25, 256, 256), dtype float16
label: 0


In [45]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image
import bitsandbytes as bnb
import accelerate
from transformers import pipeline

Import and define model id

In [46]:
model_id = "google/medgemma-1.5-4b-it"

In [47]:
# Run on the first file
from transformers import BitsAndBytesConfig
sample_file = example


model_kwargs = dict(
    device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True) # Loads the weights in 4bit pieces, allowing a larger model to fit. Speeds up inference but much slower to load initially
)

pipe = pipeline("image-text-to-text", model=model_id, model_kwargs=model_kwargs)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

In [48]:
# Zero-shot baseline helpers for MedGemma on MRNet

def get_sample(ds_split, idx):
    """Bypass HF's fixed-shape decoder so variable slice counts work."""
    table = ds_split.data.table
    return {
        "sagittal": np.array(table.column("sagittal")[idx].as_py()),
        "coronal":  np.array(table.column("coronal")[idx].as_py()),
        "axial":    np.array(table.column("axial")[idx].as_py()),
        "label":    int(table.column("label")[idx].as_py()),
    }


def slice_to_pil(mri_stack, slice_idx=None):
    """Convert one slice of an MRI stack to a PIL image in [0, 255] uint8."""
    n_slices = mri_stack.shape[0]
    if slice_idx is None:
        slice_idx = n_slices // 2  # middle slice
    if not (0 <= slice_idx < n_slices):
        raise ValueError(f"slice_idx {slice_idx} out of [0, {n_slices})")

    sl = mri_stack[slice_idx].astype(np.float32)
    sl_min, sl_max = sl.min(), sl.max()
    if sl_max - sl_min < 1e-8:
        rescaled = np.zeros_like(sl, dtype=np.uint8)
    else:
        rescaled = (255.0 * (sl - sl_min) / (sl_max - sl_min)).astype(np.uint8)
    return Image.fromarray(rescaled)


def parse_yes_no(response_text):
    """Parse MedGemma's response into 0/1. Returns None if unclear."""
    if not response_text:
        return None
    txt = response_text.strip().lower()
    words = txt.split()
    if not words:
        return None
    first = words[0].strip(".,:;!?'\"")
    if first == "yes":
        return 1
    if first == "no":
        return 0
    if "yes" in txt and "no" not in txt:
        return 1
    if "no" in txt and "yes" not in txt:
        return 0
    return None


def predict_acl(sample, pipe, view="sagittal"):
    """Zero-shot ACL tear prediction on one MRNet sample.

    Args:
        sample: dict from get_sample() with 'sagittal', 'coronal', 'axial', 'label'
        pipe: MedGemma image-text-to-text pipeline
        view: which view to use ('sagittal', 'coronal', or 'axial')
    """
    stack = sample[view]
    image = slice_to_pil(stack)  # middle slice

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "You are looking at a knee MRI slice. Does this image "
                        "show evidence of an ACL (Anterior Cruciate Ligament) "
                        "tear? Answer with a single word: 'Yes' or 'No'."
                    ),
                },
                {"type": "image", "image": image},
            ],
        }
    ]

    with torch.no_grad():
        output = pipe(text=messages, max_new_tokens=10, do_sample=False)
        response = output[0]["generated_text"][-1]["content"]

    return {
        "response": response,
        "prediction": parse_yes_no(response),
        "label": sample["label"],
        "view": view,
    }


In [49]:
# Sanity check: run on ONE sample before committing to a full eval.
sample = get_sample(ds["test"], 0)
result = predict_acl(sample, pipe, view="sagittal")
print(f"Label:        {result['label']}")
print(f"Prediction:   {result['prediction']}")
print(f"Raw response: {result['response']!r}")

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Label:        0
Prediction:   0
Raw response: 'No.\n'


In [50]:
# Zero-shot baseline eval: MedGemma on MRNet test set (no fine-tuning)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from tqdm.auto import tqdm


def evaluate_baseline(pipe, test_ds, n_samples=30, view="sagittal", seed=42, verbose=True):
    n = min(n_samples, len(test_ds))
    indices = np.random.RandomState(seed).choice(len(test_ds), size=n, replace=False)

    results = []
    for idx in tqdm(indices, desc=f"Zero-shot eval ({view})"):
        sample = get_sample(test_ds, int(idx))
        try:
            r = predict_acl(sample, pipe, view=view)
            r["idx"] = int(idx)
            results.append(r)
            if verbose:
                print(f"[{idx}] label={r['label']} pred={r['prediction']} | {r['response'][:60]}")
        except Exception as e:
            print(f"[{idx}] ERROR: {e}")
            results.append({"idx": int(idx), "prediction": None,
                            "label": sample["label"], "response": f"ERROR: {e}",
                            "view": view})

    y_true, y_pred = [], []
    n_unparsed = 0
    for r in results:
        y_true.append(int(r["label"]))
        if r["prediction"] is None:
            n_unparsed += 1
            y_pred.append(1 - int(r["label"]))  # unparseable counts as wrong
        else:
            y_pred.append(int(r["prediction"]))

    metrics = {
        "n_evaluated": len(y_true),
        "n_unparseable": n_unparsed,
        "view": view,
        "accuracy":  float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall":    float(recall_score(y_true, y_pred, zero_division=0)),
        "f1":        float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }

    print("\n" + "=" * 60)
    print(f"ZERO-SHOT BASELINE — MedGemma (no fine-tuning), view={view}")
    print("=" * 60)
    print(f"N evaluated:      {metrics['n_evaluated']}")
    print(f"Unparseable:      {metrics['n_unparseable']}")
    print(f"Accuracy:         {metrics['accuracy']:.3f}")
    print(f"Precision:        {metrics['precision']:.3f}")
    print(f"Recall:           {metrics['recall']:.3f}")
    print(f"F1:               {metrics['f1']:.3f}")
    print(f"Confusion matrix: {metrics['confusion_matrix']}")
    print("  (rows = true [no-tear, tear], cols = predicted [no-tear, tear])")
    print()
    print(classification_report(y_true, y_pred, target_names=["no-tear", "tear"],
                                zero_division=0))
    return results, metrics


# Run on 30 samples, sagittal view
results, metrics = evaluate_baseline(pipe, ds["test"], n_samples=30, view="sagittal")

Zero-shot eval (sagittal):   0%|          | 0/30 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[44] label=1 pred=0 | No.



Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[47] label=1 pred=0 | No.



...

[114] label=0 pred=0 | No.



Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[76] label=1 pred=0 | No.



Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[97] label=0 pred=0 | No.


ZERO-SHOT BASELINE — MedGemma (no fine-tuning), view=sagittal
N evaluated:      30
Unparseable:      0
Accuracy:         0.533
Precision:        0.000
Recall:           0.000
F1:               0.000
Confusion matrix: [[16, 0], [14, 0]]
  (rows = true [no-tear, tear], cols = predicted [no-tear, tear])

              precision    recall  f1-score   support

     no-tear       0.53      1.00      0.70        16
        tear       0.00      0.00      0.00        14

    accuracy                           0.53        30
   macro avg       0.27      0.50      0.35        30
weighted avg       0.28      0.53      0.37        30



In [51]:
import json

with open("baseline_results_sagittal.json", "w") as f:
    json.dump({
        "model": model_id,
        "fine_tuned": False,
        "view": "sagittal",
        "n_samples": 30,
        "metrics": metrics,
    }, f, indent=2)

print("Saved.")
print(json.dumps(metrics, indent=2))

Saved.
{
  "n_evaluated": 30,
  "n_unparseable": 0,
  "view": "sagittal",
  "accuracy": 0.5333333333333333,
  "precision": 0.0,
  "recall": 0.0,
  "f1": 0.0,
  "confusion_matrix": [
    [
      16,
      0
    ],
    [
      14,
      0
    ]
  ]
}
